In [1]:
import polars as pl
import numpy as np
import pandas as pd

from ohlc_dss_model.data import (
    load_parquet, remove_incomplete_days, write_parquet,
    intraday_session_tagging, session_tagging, 
    filter_valid_sessions
)

from ohlc_dss_model.features import(
    aggregate_sessions, yang_zhang,
    PRE_NY_SPEC, FULL_DAY_SPEC,
    calculate_excursion_bands, assign_direction,
    detect_pivots, pivot_extraction, build_pivot_features,
    build_event_table, encode_news_context, inspect_event_table,
    get_calendar_index
)

from ohlc_dss_model.utils import (
    convert_to_timezone, plot_session
)

import os
from dotenv import load_dotenv

from ohlc_dss_model.config import config

from datetime import date

In [2]:
raw_data = load_parquet()
raw_data = convert_to_timezone(raw_data)
raw_data = session_tagging(raw_data)
raw_data = intraday_session_tagging(raw_data)
raw_data = remove_incomplete_days(raw_data)

# aggregated_data = aggregate_sessions(raw_data)
# aggregated_data = filter_valid_sessions(aggregated_data)

# aggregated_data = aggregated_data.with_columns(
#     pl.col("O_Pre_Target_1").alias("O_Ref")
# )

# aggregated_data = yang_zhang(aggregated_data, FULL_DAY_SPEC, mode="historical")
# aggregated_data = yang_zhang(aggregated_data, PRE_NY_SPEC, mode="today")

# aggregated_data = assign_direction(aggregated_data)
# aggregated_data = calculate_excursion_bands(aggregated_data)
aggregated_data = load_parquet(config.data.raw_folder_path / "cache" / "fomc_load.parquet")
# aggregated_data = aggregated_data.drop_nulls()

pivots_data = detect_pivots(raw_data, n=1)
pivots_data = pivot_extraction(pivots_data)
pivots_data = build_pivot_features(pivots_data, aggregated_data)

In [3]:
print(pivots_data.columns)
pivots_data.filter(pl.col("Session") == date(2026, 2, 4)).select([
    "DateTime", "delta_Pi_k", "delta_b_k", "Speed_k", "Dir_k", "Turn_k", "P_k", "Pi_k"
]).head(8)

['DateTime', 'Intraday_Session', 'Session', 'P_k', 's_k', 'O_Pre_Target_2', 'O_Pre_Target_1', 'O_Target_1', 'O_Target_2', 'H_Pre_Target_2', 'H_Pre_Target_1', 'H_Target_1', 'H_Target_2', 'L_Pre_Target_2', 'L_Pre_Target_1', 'L_Target_1', 'L_Target_2', 'C_Pre_Target_2', 'C_Pre_Target_1', 'C_Target_1', 'C_Target_2', 'O_Ref', 'Sigma_Historical', 'Sigma_Today', 'Z_Body', 'Z_Sigma', 'Tau', 'Direction', '_delta_t', 'Band_AE_Pos_Center', 'Band_AE_Neg_Center', 'Band_FE_Pos_Center', 'Band_FE_Neg_Center', 'Band_AE_Neg_Upper', 'Band_AE_Neg_Lower', 'Band_AE_Pos_Upper', 'Band_AE_Pos_Lower', 'Band_FE_Neg_Upper', 'Band_FE_Neg_Lower', 'Band_FE_Pos_Upper', 'Band_FE_Pos_Lower', 'vix_t1', 'us10y_t1', 'us2y_t1', 'effr_t1', '10y_2y_spread_t1', 'vix_5d_delta', 'us10y_5d_delta', 'vix_pct_rank_1y_t1', 'Join_date', 'e_today', 'e_yesterday', 'e_tomorrow', 'is_fomc_day', 'is_fomc_week', 'days_to_fomc', 'is_nfp_day', 'is_cpi_day', 'is_core_cpi_day', 'Sigma_Historical_Shifted', 'Pi_k', 'Sigma_Price', 'Delta_FE_Pos',

DateTime,delta_Pi_k,delta_b_k,Speed_k,Dir_k,Turn_k,P_k,Pi_k
"datetime[ns, America/New_York]",f64,i16,f64,i32,i8,f64,f64
2026-02-03 19:00:00 EST,0.0,0,0.0,0,0,25435.75,0.09453
2026-02-03 19:30:00 EST,-0.11755,1,-0.11755,-1,0,25375.75,-0.02302
2026-02-03 21:00:00 EST,0.191998,3,0.063999,1,1,25473.75,0.168978
2026-02-03 21:30:00 EST,-0.145468,1,-0.145468,-1,1,25399.5,0.02351
2026-02-03 22:30:00 EST,0.102366,2,0.051183,1,1,25451.75,0.125876
2026-02-03 23:30:00 EST,-0.007347,2,-0.003673,-1,1,25448.0,0.118529
2026-02-04 00:30:00 EST,-0.076897,2,-0.038449,-1,0,25408.75,0.041632
2026-02-04 02:30:00 EST,0.206692,4,0.051673,1,1,25514.25,0.248324


***
### **Dataset Contract**
This section will be utilized the transform the already engineered feature datas into a dataset for transformer.

Before moving on we will be defining which features are to be included in the dataset this is done for us not to have any headache later

In [4]:
# Defining pivot column whitelists

# configurable in config.py
MAX_PIVOT = 27

PIVOT_NUMERIC_WHITELIST = [
    "Pi_k",
    'Delta_FE_Pos', 'Delta_AE_Pos', 'Delta_FE_Neg', 'Delta_AE_Neg',
    'State_AE_Neg', 'State_AE_Pos', 'State_FE_Neg', 'State_FE_Pos',
    "delta_Pi_k", "delta_b_k", "Speed_k", "Dir_k", "Turn_k",
]

PIVOT_CATEGORICAL_WHITELIST = ["s_k", "Intraday_Session"]

In [5]:
_df = aggregated_data.with_columns(
        pl.col("Sigma_Historical").shift(1).alias("Sigma_Historical_Shifted")
    )

#### **Normalizing Session OHLC Data**
Since we are feeding it into the context hence we will need to normalize it so it is consistent across all regime

$$\sigma_{Price} = \sigma_{YZ}(d-1) * O_Ref$$
$$\tilde{X}_{d}^{(s)} = \frac{X_{d}^{(s)} - O_{Ref,d}}{\sigma_{\text{Price}}}$$

where $X \in \{H, L, C, O\}$ and $s \in \{\text{Asia},\ \text{London}\}$.

In [6]:
# sigma_price = pl.col("Sigma_Historical_Shifted") * pl.col("O_Ref")
# _df = _df.with_columns([
#     ((pl.col("H_Pre_Target_1") - pl.col("O_Ref")) / sigma_price).alias("H_Pre_Target_1_Normalized"),
#     ((pl.col("L_Pre_Target_1") - pl.col("O_Ref")) / sigma_price).alias("L_Pre_Target_1_Normalized"),
#     ((pl.col("C_Pre_Target_1") - pl.col("O_Ref")) / sigma_price).alias("C_Pre_Target_1_Normalized"),
#     ((pl.col("O_Pre_Target_2") - pl.col("O_Ref")) / sigma_price).alias("O_Pre_Target_2_Normalized"),
#     ((pl.col("H_Pre_Target_2") - pl.col("O_Ref")) / sigma_price).alias("H_Pre_Target_2_Normalized"),
#     ((pl.col("L_Pre_Target_2") - pl.col("O_Ref")) / sigma_price).alias("L_Pre_Target_2_Normalized"),
#     ((pl.col("C_Pre_Target_2") - pl.col("O_Ref")) / sigma_price).alias("C_Pre_Target_2_Normalized"),
# ])

***
#### **Economic Event Encoding**

For each trading day $d$, three event context features are constructed capturing the macro event weight on the preceding, current, and following calendar day:

$$\mathbf{n}_d = \left[\, e_{d-1},\ e_d,\ e_{d+1} \,\right] \in \{0, 1, 2, 3\}^3$$

where the daily event weight $e_t$ is defined as the maximum impact weight across all qualifying USD events on day $t$:

$$e_t = \max_{i \in \mathcal{E}_t} w_i, \qquad e_t = 0 \text{ if } \mathcal{E}_t = \emptyset$$

with impact weight tiers:

$$w_i = \begin{cases} 3 & \text{ultra-high: CPI, Core CPI, NFP, FOMC} \\ 2 & \text{high: Unemployment Claims, Average Hourly Earnings} \\ 1 & \text{medium: PPI, Core PPI, ADP, ISM Manufacturing,} \\ & \phantom{\text{medium: }} \text{ISM Services, JOLTs, Core Retail Sales} \\ 0 & \text{no qualifying event} \end{cases}$$

In [7]:
# load_dotenv()
# event_table = build_event_table(
#     aggregated_data.select(pl.col("Session").min()).item(),
#     aggregated_data.select(pl.col("Session").max()).item(),
#     os.getenv("ALFRED_API_KEY")
# )
# write_parquet(event_table, "event_table")

In [8]:
# inspect = inspect_event_table(os.getenv("ALFRED_API_KEY"), date(2016, 1, 5), date(2016, 2, 10))
# print(inspect.head(10))

In [9]:
# event_table = load_parquet(config.data.event_path)
# _df = encode_news_context(_df, event_table)

In [10]:
# event_table.head(5)

In [11]:
# _df.filter(pl.col("Session") == date(2026, 2, 12)).select(["Session", "e_yesterday", "e_today", "e_tomorrow"])

In [12]:
_df = get_calendar_index(_df)

***
#### **Context Features**

In [13]:
sigma_price = pl.col("Sigma_Historical_Shifted") * pl.col("O_Ref")
eps = 1e-9
_df = _df.with_columns([
    ((pl.col("H_Pre_Target_2") - pl.col("L_Pre_Target_2")) / sigma_price).alias("ps2_range_norm"),

    ((pl.col("H_Pre_Target_1") - pl.col("L_Pre_Target_1")) / sigma_price).alias("ps1_range_norm"),

    ((pl.col("C_Pre_Target_2") - pl.col("L_Pre_Target_2")) /
     (pl.col("H_Pre_Target_2") - pl.col("L_Pre_Target_2") + eps))
    .alias("ps2_close_pct_within_ps2_range"),

    ((pl.col("C_Pre_Target_1") - pl.col("L_Pre_Target_1")) /
     (pl.col("H_Pre_Target_1") - pl.col("L_Pre_Target_1") + eps))
    .alias("ps1_close_pct_within_ps1_range"),

    ((pl.col("O_Target_1") - pl.col("C_Pre_Target_2")) / sigma_price)
    .alias("target_open_gap_norm"),

    (pl.col("C_Pre_Target_1") > pl.col("O_Pre_Target_1")).alias("_ps1_bull"),
    (pl.col("C_Pre_Target_2") > pl.col("O_Pre_Target_2")).alias("_ps2_bull"),
]).with_columns([
    (pl.col("ps2_range_norm") / (pl.col("ps1_range_norm") + eps))
    .alias("ps2_vs_ps1_range_ratio"),

    (pl.col("_ps1_bull") == pl.col("_ps2_bull")).alias("sessions_agree"),
]).drop(["_ps1_bull", "_ps2_bull"])

In [14]:
# sigma_vals = aggregated_data["Sigma_Today"].to_numpy().astype(float)
# aggregated_data = aggregated_data.with_columns([
#     pl.Series("sigma_today_pct_rank_20d", _numpy_rolling_pct_rank(sigma_vals, 20)),
#     pl.Series("sigma_today_pct_rank_60d", _numpy_rolling_pct_rank(sigma_vals, 60)),
# ])

In [15]:
_df = _df.with_columns([
    pl.col("C_Target_2").shift(1).alias("_prior_close"),
]).with_columns([
    pl.col("_prior_close").rolling_mean(20).alias("_ma20"),
    pl.col("_prior_close").rolling_mean(200).alias("_ma200"),
    (pl.col("_prior_close") / pl.col("_prior_close").shift(5) - 1.0).alias("nq_5d_return"),
    (pl.col("_prior_close") / pl.col("_prior_close").shift(20) - 1.0).alias("nq_20d_return"),
]).with_columns([
    (pl.col("_prior_close") > pl.col("_ma20")).alias("nq_above_20d_ma"),
    (pl.col("_prior_close") > pl.col("_ma200")).alias("nq_above_200d_ma"),
    ((pl.col("_prior_close") - pl.col("_ma20")) /
     (pl.col("Sigma_Historical") * pl.col("O_Ref") + 1e-9))
    .alias("nq_dist_from_20d_ma_norm"),
]).drop(["_prior_close", "_ma20", "_ma200"])

In [16]:
_df = _df.with_columns(
    pl.when(pl.col("C_Pre_Target_2") > pl.col("Band_FE_Pos_Upper")).then(pl.lit(6))
    .when(pl.col("C_Pre_Target_2") >= pl.col("Band_FE_Pos_Lower")).then(pl.lit(4))
    .when(pl.col("C_Pre_Target_2") >= pl.col("Band_AE_Pos_Upper")).then(pl.lit(2))
    .when(pl.col("C_Pre_Target_2") > pl.col("Band_AE_Neg_Upper")).then(pl.lit(1))
    .when(pl.col("C_Pre_Target_2") >= pl.col("Band_AE_Neg_Lower")).then(pl.lit(1))
    .when(pl.col("C_Pre_Target_2") >= pl.col("Band_FE_Neg_Upper")).then(pl.lit(3))
    .when(pl.col("C_Pre_Target_2") >= pl.col("Band_FE_Neg_Lower")).then(pl.lit(5))
    .otherwise(pl.lit(7))
    .alias("band_state_ps2")
)

***

The context table $\mathcal{C}$ is indexed by trading day $d \in \mathcal{D}$:

$$\mathcal{C}_d = \left[\,\tilde{H}_d^{A},\ \tilde{L}_d^{A},\ \tilde{C}_d^{A},\ \tilde{O}_d^{L},\ \tilde{H}_d^{L},\ \tilde{L}_d^{L},\ \tilde{C}_d^{L},\ \sigma_d,\ \sigma_{d-1},\ e_{d-1}, \ e_{d}, \ e_{d+1} \ \right] \in \mathbb{R}^{12}$$

where $d$ corresponds to the `Session` date key used for alignment with pivot sequences and targets.

In [17]:
# Columns that will go in context table

CONTEXT_WHITELIST = [
    "Sigma_Today", "Sigma_Historical_Shifted",
    "e_yesterday", "e_today", "e_tomorrow",
    # "H_Pre_Target_1_Normalized", "L_Pre_Target_1_Normalized", "C_Pre_Target_1_Normalized",
    # "O_Pre_Target_2_Normalized", "H_Pre_Target_2_Normalized",
    # "L_Pre_Target_2_Normalized", "C_Pre_Target_2_Normalized",
    "vix_t1", "us10y_t1", "us2y_t1", "effr_t1", "10y_2y_spread_t1",
    "vix_5d_delta", "us10y_5d_delta", "vix_pct_rank_1y_t1", "is_fomc_day",
    "is_fomc_week", "days_to_fomc", "is_nfp_day", "is_cpi_day", "is_core_cpi_day",
    "day_of_week", "month", "week_of_month",
    "ps2_range_norm", "ps1_range_norm", "ps2_close_pct_within_ps2_range", "ps1_close_pct_within_ps1_range", "ps2_vs_ps1_range_ratio", "sessions_agree",
    "nq_5d_return", "nq_20d_return", "nq_above_20d_ma", "nq_above_200d_ma", "nq_dist_from_20d_ma_norm",
    "band_state_ps2"
    
]

In [18]:
ctx_df = (
    _df.select(["Session", *CONTEXT_WHITELIST])
    .drop_nulls(["Session", *CONTEXT_WHITELIST])
    .sort("Session")
)

In [19]:
print(ctx_df.columns)

['Session', 'Sigma_Today', 'Sigma_Historical_Shifted', 'e_yesterday', 'e_today', 'e_tomorrow', 'vix_t1', 'us10y_t1', 'us2y_t1', 'effr_t1', '10y_2y_spread_t1', 'vix_5d_delta', 'us10y_5d_delta', 'vix_pct_rank_1y_t1', 'is_fomc_day', 'is_fomc_week', 'days_to_fomc', 'is_nfp_day', 'is_cpi_day', 'is_core_cpi_day', 'day_of_week', 'month', 'week_of_month', 'ps2_range_norm', 'ps1_range_norm', 'ps2_close_pct_within_ps2_range', 'ps1_close_pct_within_ps1_range', 'ps2_vs_ps1_range_ratio', 'sessions_agree', 'nq_5d_return', 'nq_20d_return', 'nq_above_20d_ma', 'nq_above_200d_ma', 'nq_dist_from_20d_ma_norm', 'band_state_ps2']


In [20]:
ctx_df.filter(pl.col("Session") == date(2026, 2, 4))

Session,Sigma_Today,Sigma_Historical_Shifted,e_yesterday,e_today,e_tomorrow,vix_t1,us10y_t1,us2y_t1,effr_t1,10y_2y_spread_t1,vix_5d_delta,us10y_5d_delta,vix_pct_rank_1y_t1,is_fomc_day,is_fomc_week,days_to_fomc,is_nfp_day,is_cpi_day,is_core_cpi_day,day_of_week,month,week_of_month,ps2_range_norm,ps1_range_norm,ps2_close_pct_within_ps2_range,ps1_close_pct_within_ps1_range,ps2_vs_ps1_range_ratio,sessions_agree,nq_5d_return,nq_20d_return,nq_above_20d_ma,nq_above_200d_ma,nq_dist_from_20d_ma_norm,band_state_ps2
date,f64,f64,i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,i16,bool,bool,bool,i8,i8,i8,f64,f64,f64,f64,f64,bool,f64,f64,bool,bool,f64,i32
2026-02-04,0.00523,0.020105,0,1,2,18.0,4.28,3.57,3.64,0.71,1.65,0.04,0.622222,false,false,42,false,false,false,3,2,6,0.303671,0.298283,0.256452,0.926108,1.018062,false,-0.026379,-0.006138,false,true,-0.58648,1


***
#### **Label Construction**
***
**Normalized Excursions**

This measures how far price moved up and down from NY Open, to account for regime changes it would be scaled by daily volatility.
$$z_{pos} = \frac{H_{NY} - O_{NY}}{\sigma_{YZ} \cdot O_{NY}}$$
$$z_{neg} = \frac{O_{NY} - L_{NY}}{\sigma_{YZ} \cdot O_{NY}}$$


In [21]:
eps = 1e-8 # just a very small number to prevent division by zero in normalization
target_col = "Target_1"
z_pos = ((pl.col(f"H_{target_col}") - pl.col(f"O_{target_col}")).clip(lower_bound=0) / (pl.col("Sigma_Historical") * pl.col(f"O_{target_col}") + eps))
z_neg = ((pl.col(f"O_{target_col}") - pl.col(f"L_{target_col}")).clip(lower_bound=0) / (pl.col("Sigma_Historical") * pl.col(f"O_{target_col}") + eps))
label_df = (
    aggregated_data.select(["Session", f"H_{target_col}", f"O_{target_col}", f"L_{target_col}", "Sigma_Historical"]).with_columns([
        z_pos.alias("z_pos"),
        z_neg.alias("z_neg")
    ]).select(["Session", "z_pos", "z_neg"]).drop_nulls().sort("Session")
)
label_df = label_df.with_columns([
    (pl.max_horizontal(pl.col("z_pos"), pl.col("z_neg")).alias("z_max")),
])
label_df = label_df.with_columns([
    ((pl.when((pl.col("z_max") >= 0.3) | ((pl.col("z_pos") - pl.col("z_neg")).abs() >= 0.2)).then(pl.lit(0)).otherwise(pl.lit(1))).alias("ambiguous")),
    ((pl.when(pl.col("z_pos") > pl.col("z_neg")).then(pl.lit(1)).otherwise(pl.lit(-1))).alias("z_dir"))
])

In [22]:
print(ctx_df.filter(pl.col("Session") == date(2026, 2, 4)))
print(label_df.filter(pl.col("Session") == date(2026, 2, 4)))

shape: (1, 35)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Session   ┆ Sigma_Tod ┆ Sigma_His ┆ e_yesterd ┆ … ┆ nq_above_ ┆ nq_above_ ┆ nq_dist_f ┆ band_sta │
│ ---       ┆ ay        ┆ torical_S ┆ ay        ┆   ┆ 20d_ma    ┆ 200d_ma   ┆ rom_20d_m ┆ te_ps2   │
│ date      ┆ ---       ┆ hifted    ┆ ---       ┆   ┆ ---       ┆ ---       ┆ a_norm    ┆ ---      │
│           ┆ f64       ┆ ---       ┆ i8        ┆   ┆ bool      ┆ bool      ┆ ---       ┆ i32      │
│           ┆           ┆ f64       ┆           ┆   ┆           ┆           ┆ f64       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2026-02-0 ┆ 0.00523   ┆ 0.020105  ┆ 0         ┆ … ┆ false     ┆ true      ┆ -0.58648  ┆ 1        │
│ 4         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
└───────────┴───────────┴───────────┴───────────┴───┴───────────┴───────────

***
#### **Transformer Dataset**
For each session (day) $d$, we construct a token matrix:
$$X_d \in \R^{T\times D}$$

$$T = 1 + max\_pivots$$
$$D = 1 + F_c + F_p + F_{cat}$$
where:
- 1 -> context token
- max_pivots -> number of maximum tokens
- $F_c$ -> Context columns count
- $F_p$ -> Pivots numerical columns count
- $F_{cat}$ -> Pivots categorical columns count
***
**Token Structure**

Each token $x_i \in \R^D$:
$$x_i = \begin{cases}
\left[1,\ c_d, \ 0, \ 0 \right] \qquad \text{if i = 0 (context)}\\
\left[0,\ 0, \ p_k, \ e_k \right] \qquad \text{if i = 0 (context)}\\
\end{cases}
$$
where:
- $c_d$ -> Context features
- $p_k$ -> Pivot numerical features
- $e_k$ -> Pivot categorical features

In [23]:
required = ["Session", *PIVOT_NUMERIC_WHITELIST, *PIVOT_CATEGORICAL_WHITELIST]
pivots_df = pivots_data.select(required)
pivots_df = pivots_df.with_columns(pl.int_range(pl.len()).over("Session").alias("global_pos"))

In [24]:
valid_sessions = (
    ctx_df.select("Session")
    .join(label_df.select("Session"), on="Session", how="inner")
    .join(pivots_df.select("Session").unique(), on="Session", how="inner")
    .sort("Session")
)
# to skip burn in period
valid_sessions = valid_sessions.filter(pl.col("Session") >= date(2011, 4, 5))["Session"].to_list()

In [25]:
print(valid_sessions[:5])

[datetime.date(2011, 10, 21), datetime.date(2011, 10, 24), datetime.date(2011, 10, 25), datetime.date(2011, 10, 26), datetime.date(2011, 10, 27)]


In [26]:
pivots_df.filter(pl.col("Session") == date(2026, 2, 4)).tail(5)

Session,Pi_k,Delta_FE_Pos,Delta_AE_Pos,Delta_FE_Neg,Delta_AE_Neg,State_AE_Neg,State_AE_Pos,State_FE_Neg,State_FE_Pos,delta_Pi_k,delta_b_k,Speed_k,Dir_k,Turn_k,s_k,Intraday_Session,global_pos
date,f64,f64,f64,f64,f64,i8,i8,i8,i8,f64,i16,f64,i32,i8,i32,str,i64
2026-02-04,0.041632,-0.996157,-0.303093,1.079422,0.386358,0,0,0,0,-0.076897,2,-0.038449,-1,0,-1,"""Pre_Target_1""",6
2026-02-04,0.248324,-0.789465,-0.096402,1.286114,0.59305,0,1,0,0,0.206692,4,0.051673,1,1,1,"""Pre_Target_1""",7
2026-02-04,-0.064652,-1.102442,-0.409378,0.973137,0.280073,0,0,0,0,-0.312977,4,-0.078244,-1,1,-1,"""Pre_Target_2""",8
2026-02-04,0.154284,-0.883505,-0.190441,1.192074,0.49901,0,0,0,0,0.218937,2,0.109468,1,1,1,"""Pre_Target_2""",9
2026-02-04,0.201794,-0.835996,-0.142932,1.239584,0.54652,0,0,0,0,0.04751,3,0.015837,1,0,1,"""Pre_Target_2""",10


In [27]:
intraday_session_map = {"Pre_Target_1": 1, "Pre_Target_2": 2}
s_k_map = {-1: 0, 1: 1}

Fc = len(CONTEXT_WHITELIST)
Fp = len(PIVOT_NUMERIC_WHITELIST)
Fcat = len(PIVOT_CATEGORICAL_WHITELIST)

max_pivots = 27

# 1 here is accounting for context flag
token_dim = 1 + Fc + Fp + Fcat
T = 1 + max_pivots


***
**Session Matrix**

For each session $d$:
$$X_{d} = 
\begin{bmatrix}
1 & c_1 & \cdot\cdot\cdot & c_{F_{c}} & 0 & \cdot\cdot\cdot & 0 & 0 & \cdot\cdot\cdot & 0\\
0 & 0 & \cdot\cdot\cdot & 0 & p_{1,1} & \cdot\cdot\cdot & p_{1, j} & e_{1, 1} & \cdot\cdot\cdot & e_{1, k}\\
0 & 0 & \cdot\cdot\cdot & 0 & p_{1,1} & \cdot\cdot\cdot & p_{2, N} & e_{2, 1} & \cdot\cdot\cdot & e_{2, k}\\
\vdots & \vdots & & \vdots & \vdots & & \vdots & \vdots & & \vdots\\
0 & 0 & \cdot\cdot\cdot & 0 & p_{i,1} & \cdot\cdot\cdot & p_{i, j} & e_{i, 1} & \cdot\cdot\cdot & e_{i, k}\\
\vdots & \vdots & & \vdots & \vdots & & \vdots & \vdots & & \vdots\\
\end{bmatrix}
$$

where:
- $p_{i, j}$ = Pivots numerical values
- $e_{i, k}$ = Pivots categorical values

<br>

**Attention Mask**:
$$m_d \in \{ 0,1\}^T$$
$$m_i = \begin{cases}
1 \qquad token\\
0 \qquad padding
\end{cases}
$$
This mask is to distinguish which are real tokens and which are just padding, we use padding since our number of pivots are not fixed.

<br>

**Position**:

For each session:
$$p_d \in \N^T$$
$$p_i = \begin{cases}
0 \qquad \text{Context Token}\\
k \qquad \text{Pivot Index}\\
0 \qquad \text{Padding}
\end{cases}
$$

These are positions for tokens inside a single session window

<br>

**Labels**:

$$y_d = [z_{pos}, \ z_{neg}]$$

where:
- $z_{pos}$ = Normalized positive excursion from $O_{NY}$
- $z_{neg}$ = Normalized negative excursion from $O_{NY}$

In [28]:
# diagnostics variables
max_seen_pivots = 0
truncated_days = 0

In [29]:
x_tokens, mask, position = [], [], []
z_max_labels, z_dir_labels, ambiguous_labels = [], [], []
sessions = []

In [30]:
for sess in valid_sessions:
    c_row = ctx_df.filter(pl.col("Session") == sess)
    p_day = pivots_df.filter(pl.col("Session") == sess)
    l_row = label_df.filter(pl.col("Session") == sess)

    if c_row.height != 1 or l_row.height != 1:
        continue

    max_seen_pivots = max(max_seen_pivots, p_day.height)
    if p_day.height > max_pivots:
        truncated_days += 1
        # if max pivots then older pivots are truncated
        p_day = p_day.tail(max_pivots)

    x = np.zeros((T, token_dim), dtype=np.float32)
    m = np.zeros((T,), dtype=np.int64)
    p = np.zeros((T,), dtype=np.int64)

    ctx_vals = c_row.select(CONTEXT_WHITELIST).to_numpy().astype(np.float32)[0]
    # flag for context row
    x[0, 0] = 1.0

    x[0, 1:1+Fc] = ctx_vals
    m[0] = 1
    p[0] = 0

    i = 1
    for row in p_day.iter_rows(named=True):
        if i >= T:
            break

        p_num = np.array([float(row[col]) for col in PIVOT_NUMERIC_WHITELIST], dtype=np.float32)
        s_k_encoded = s_k_map.get(int(row["s_k"]), 0)
        intraday_session_encoded = intraday_session_map.get(str(row["Intraday_Session"]), 0)

        start = 1 + Fc
        x[i, start:start+Fp] = p_num
        x[i, start+Fp] = float(s_k_encoded)
        x[i, start+Fp+1] = float(intraday_session_encoded)

        m[i] = 1
        p[i] = int(row["global_pos"]) + 1
        i += 1
    x_tokens.append(x)
    mask.append(m)
    position.append(p)
    z_max_labels.append(float(l_row["z_max"][0]))
    z_dir_labels.append(int(l_row["z_dir"][0]))
    ambiguous_labels.append(int(l_row["ambiguous"][0]))
    sessions.append(sess)

In [31]:
dataset = {
    "X_Tokens": np.stack(x_tokens, axis=0),
    "Attention_Mask": np.stack(mask, axis=0),
    "Token_Position": np.stack(position, axis=0),
    "Z_Max_Labels": np.array(z_max_labels, dtype=np.float32),
    "Z_Dir_Labels": np.array(z_dir_labels, dtype=np.int64),
    "Z_Ambiguous_Labels": np.array(ambiguous_labels, dtype=np.int64),
    "Sessions": sessions,
    "Features_Metadata": {
        "Max_Pivots": max_pivots,
        "Height": T,
        "Token_Dim": token_dim,
        "Truncated_Days": truncated_days,
        "Max_Seen_Pivots": max_seen_pivots,
    }
}

In [32]:
print(dataset["Features_Metadata"]["Max_Seen_Pivots"])

19


In [33]:
print(f"Session: {dataset['Sessions'][0]}")
print(f"Z_Max_Labels: {dataset['Z_Max_Labels'][0]}")
print(f"Z_Dir_Labels: {dataset['Z_Dir_Labels'][0]}")
print(f"Z_Ambiguous_Labels: {dataset['Z_Ambiguous_Labels'][0]}")

Session: 2011-10-21
Z_Max_Labels: 0.4664563536643982
Z_Dir_Labels: 1
Z_Ambiguous_Labels: 0


***